In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"

CLEAN_DATA_PATH = OUTPUT_DIR / "APL_Logistics_cleaned.csv"

In [3]:
df = pd.read_csv(
    CLEAN_DATA_PATH,
    encoding="latin1"
)

In [4]:
print("Rows:",df.shape[0])
print("Columns:",df.shape[1])

Rows: 180519
Columns: 40


In [5]:
TARGET = "Late_delivery_risk"
df[TARGET].value_counts()

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

In [6]:
LEAKAGE_COLUMNS = [
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Delivery Status",
    "Order Status"
]

In [7]:
PII_COLUMNS = [
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Customer Id",
    "Order Customer Id",
    "Customer Zipcode"
]

In [8]:
missing_leakage = [
    col for col in LEAKAGE_COLUMNS
    if col not in df.columns
]

missing_pii = [
    col for col in PII_COLUMNS
    if col not in df.columns
]

print("Missing leakage columns:",missing_leakage)
print("Missing PII columns:",missing_pii)

Missing leakage columns: []
Missing PII columns: []


In [9]:
df["discount_amount_per_unit"] = (
    df["Order Item Discount"] 
    / df["Order Item Quantity"].clip(lower=1)
)

In [10]:
df["sales_per_quantity"] = (
    df["Sales"] 
    / df["Order Item Quantity"].clip(lower=1)
)

In [11]:
df["profit_per_quantity"] = (
    df["Order Profit Per Order"]
    / df["Order Item Quantity"].clip(lower=1)
)

In [12]:
df["price_discount_interaction"] = (
    df["Order Item Product Price"]
    * df["Order Item Discount Rate"]
)

In [13]:
df["scheduled_days_bucket"] = (
    df["Days for shipment (scheduled)"]
    .astype(str)
)

In [14]:
df["geo_market_region"] = (
    df["Market"].astype(str)
    + " | "
    + df["Order Region"].astype(str)
)

In [15]:
new_features = [
    "discount_amount_per_unit",
    "sales_per_quantity",
    "profit_per_quantity",
    "price_discount_interaction",
    "scheduled_days_bucket",
    "geo_market_region"
]

df[new_features].head()

,discount_amount_per_unit,sales_per_quantity,profit_per_quantity,price_discount_interaction,scheduled_days_bucket,geo_market_region
0,5.500,99.99,31.938,5.9994,4,Pacific Asia | South Asia
1,6.398,39.99,9.742,6.3984,4,LATAM | Central America
2,18.000,199.99,87.360,17.9991,4,LATAM | Central America
3,24.000,199.99,-41.890,23.9988,4,USCA | East of USA
4,10.000,50.00,10.000,10.0000,4,USCA | East of USA


In [16]:
np.isinf(
    df.select_dtypes(include=np.number)
).sum().sum()

np.int64(0)

In [17]:
df[new_features].isnull().sum()

discount_amount_per_unit      0
sales_per_quantity            0
profit_per_quantity           0
price_discount_interaction    0
scheduled_days_bucket         0
geo_market_region             0
dtype: int64

In [18]:
DROP_COLUMNS = LEAKAGE_COLUMNS + PII_COLUMNS

In [19]:
ml_df = df.drop(
    columns = DROP_COLUMNS,
    errors = "ignore"
).copy()

In [20]:
ml_df.shape

(180519, 36)

In [21]:
x = ml_df.drop(
    columns=[TARGET],
    errors="ignore"
)

y=df[TARGET].astype(int)

In [22]:
print("x shape:",x.shape)
print("y shape:",y.shape)

x shape: (180519, 36)
y shape: (180519,)


In [23]:
TARGET in x.columns

False

In [24]:
remaining_leakage = [
    col for col in LEAKAGE_COLUMNS
    if col in x.columns
]

remaining_leakage

[]

In [25]:
remaining_pii = [
    col for col in PII_COLUMNS
    if col in x.columns
]

remaining_pii

[]

In [26]:
numeric_features = x.select_dtypes(
    include = np.number
).columns.tolist()

In [27]:
len(numeric_features)

20

In [28]:
numeric_features

['Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Category Id',
 'Department Id',
 'Latitude',
 'Longitude',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Product Price',
 'discount_amount_per_unit',
 'sales_per_quantity',
 'profit_per_quantity',
 'price_discount_interaction']

In [30]:
categorical_features = x.select_dtypes(
    include=["object"]
).columns.tolist()

C:\Users\ganesh\AppData\Local\Temp\ipykernel_27772\3884464240.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(


In [31]:
len(categorical_features)

16

In [32]:
categorical_features

['Type',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Segment',
 'Customer State',
 'Department Name',
 'Market',
 'Order City',
 'Order Country',
 'Order Region',
 'Order State',
 'Product Name',
 'Shipping Mode',
 'scheduled_days_bucket',
 'geo_market_region']

In [33]:
print("Numeric features:",len(numeric_features))
print("Categorical features:",len(categorical_features))
print("Total features:",x.shape[1])

Numeric features: 20
Categorical features: 16
Total features: 36


In [34]:
cardinality= (
    x[categorical_features]
    .nunique()
    .sort_values(ascending=False)
)

cardinality

Order City               3597
Order State              1089
Customer City             563
Order Country             164
Product Name              118
Category Name              50
Customer State             46
Order Region               23
geo_market_region          23
Department Name            11
Market                      5
Type                        4
Shipping Mode               4
scheduled_days_bucket       4
Customer Segment            3
Customer Country            2
dtype: int64

In [35]:
x[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
Benefit per order,180519.0,21.974989,104.433526,-4274.980000,7.000000,31.520000,64.800000,911.800000
Sales per customer,180519.0,183.107607,120.043668,7.490000,104.380000,163.990000,247.400000,1939.990000
Category Id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
Department Id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000
Latitude,180519.0,29.719955,9.813646,-33.937553,18.265432,33.144863,39.279617,48.781933
Longitude,180519.0,-84.915675,21.433241,-158.025986,-98.446312,-76.847908,-66.370583,115.263077
Order Item Discount,180519.0,20.664741,21.800901,0.000000,5.400000,14.000000,29.990000,500.000000
Order Item Discount Rate,180519.0,0.101668,0.070415,0.000000,0.040000,0.100000,0.160000,0.250000
Order Item Product Price,180519.0,141.232547,139.732489,9.990000,50.000000,59.990000,199.990000,1999.990000


In [36]:
target_distribution = (
    y.value_counts()
    .sort_index()
)

target_distribution

Late_delivery_risk
0    81542
1    98977
Name: count, dtype: int64

In [37]:
target_percenatge = (
    y.value_counts(normalize=True)
    .sort_index()
    * 100
)

target_percenatge

Late_delivery_risk
0    45.170868
1    54.829132
Name: proportion, dtype: float64

In [38]:
ml_ready_df = x.copy()

ml_ready_df[TARGET] = y

In [39]:
ML_READY_PATH = (
    OUTPUT_DIR / "APL_Logistics_ml_ready.csv"
)

ml_ready_df.to_csv(
    ML_READY_PATH,
    index= False
)

In [41]:
feature_summary = pd.DataFrame({
    "Feature": x.columns,
    "Type": [
        "Numeric" if col in numeric_features
        else "Categorical"
        for col in x.columns
    ]
})
feature_summary.head()

,Feature,Type
0,Type,Categorical
1,Days for shipment (scheduled),Numeric
2,Benefit per order,Numeric
3,Sales per customer,Numeric
4,Category Id,Numeric


In [42]:
feature_summary.to_csv(
    OUTPUT_DIR / "feature_summary.csv",
    index=False
)

In [43]:
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)

print("Original rows:", len(df))
print("ML rows:", len(x))
print("Predictor features:", x.shape[1])
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Target:", TARGET)
print("Target missing:", y.isnull().sum())
print("Remaining leakage:", len(remaining_leakage))
print("Remaining PII:", len(remaining_pii))
print("ML-ready file:", ML_READY_PATH)

print("=" * 60)

FEATURE ENGINEERING COMPLETE
Original rows: 180519
ML rows: 180519
Predictor features: 36
Numeric features: 20
Categorical features: 16
Target: Late_delivery_risk
Target missing: 0
Remaining leakage: 0
Remaining PII: 0
ML-ready file: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs\APL_Logistics_ml_ready.csv
